# BWF Player Lookup

Enter a player name to find their bwfbadminton.com profile, personal details and ranking.

_Search (Iteration 1), personal details (Iteration 2) and ranking (Iteration 3) are implemented. Final polish arrives in Iteration 4._

In [ ]:
import logging

from bwf_player import BlockedByCloudflareError, BwfClientError
from bwf_player.profile import get_profile
from bwf_player.ranking import get_ranking
from bwf_player.search import search_player

logging.basicConfig(level=logging.INFO)

In [ ]:
PLAYER_NAME = "jonathan cristie"

try:
    result = search_player(PLAYER_NAME)
except BlockedByCloudflareError as exc:
    raise SystemExit(f"Blocked by Cloudflare - wait before retrying. {exc}")
except BwfClientError as exc:
    raise SystemExit(f"Request failed: {exc}")

print(result.status.upper(), "-", result.message)
if result.best_match:
    print(result.best_match.profile_url)
for c in result.candidates:
    print(f"  {c.score:5.1f}  {c.name} ({c.country or '?'})  {c.profile_url}")

In [ ]:
if result.best_match:
    try:
        profile = get_profile(result.best_match.player_id)
    except (BlockedByCloudflareError, BwfClientError) as exc:
        raise SystemExit(f"Request failed: {exc}")

    show = lambda value, unit="": "null" if value is None else f"{value}{unit}"
    print(f"Name:         {show(profile.name)}")
    print(f"Nationality:  {show(profile.nationality)}")
    print(f"Height:       {show(profile.height_cm, ' cm')}")
    print(f"Playing hand: {show(profile.playing_hand)}")
    for note in profile.notes:
        print("  note:", note)
else:
    print("No single player selected - pick one of the candidates above and re-run with a fuller name.")

In [ ]:
if result.best_match:
    try:
        ranking = get_ranking(result.best_match.player_id)  # add event_id="9-..." to pick another event
    except (BlockedByCloudflareError, BwfClientError) as exc:
        raise SystemExit(f"Request failed: {exc}")

    print(f"Event:          {ranking.event.name if ranking.event else 'null'}")
    print(f"Current rank:   {ranking.current_rank if ranking.is_ranked else 'null'}")
    if ranking.weeks_at_current_rank is not None:
        print(f"At this rank:   {ranking.weeks_at_current_rank} week(s), since {ranking.at_rank_since} (latest list {ranking.as_of})")
    else:
        print("At this rank:   null")
    if ranking.other_events:
        print("Other events:  ", ", ".join(f"{e.name} [{e.id}]" for e in ranking.other_events))
    for note in ranking.notes:
        print("  note:", note)